# W25 · ros2_control：标准化的控制框架

> 到上一讲为止，你直接用 `cmd_vel` 推 Gazebo 插件——快但不可移植。
> ros2_control 把「控制器 ↔ 硬件」抽象成统一接口：同一份控制器配置，
> 今天接 Gazebo，明天接真实电机驱动，控制器代码一行不改。
> 对 RL 工程师，它是「策略输出 → 关节力矩/速度」之间的标准传送带。

## 学习目标

1. 画出 ros2_control 架构图：ControllerManager / ResourceManager / Hardware Interface / Controller；
2. 会在 URDF 中写 `<ros2_control>` 标签声明关节的 command/state interface；
3. 会配置并启动控制器链：`joint_state_broadcaster` + `diff_drive_controller`；
4. 理解控制器的生命周期（unconfigured → inactive → active）与 `spawner`；
5. 理解差速控制的运动学，为 W27「RL 策略接管 cmd_vel」做准备。

## ⚠️ 运行前提

配置与命令需 ROS2 Jazzy + `ros-jazzy-ros2-control ros-jazzy-ros2-controllers
ros-jazzy-gz-ros2-control`。运动学模拟为纯 Python，本机真实执行。

## 1. 架构：插件化的「控制总线」

```
        你的节点 (Nav2 / RL 策略)
              │ cmd_vel / 轨迹
              ▼
   ┌────────────────────── ControllerManager ────────────────────┐
   │  controller A      controller B       controller C           │
   │  (diff_drive)      (joint_state_      (position_trajectory)  │
   │                    broadcaster)                              │
   └────────┬───────────────────────────────────┬────────────────┘
            │ command interface                 │ state interface
            ▼                                   ▲
   ┌──────────────── ResourceManager ────────────────────────────┐
   │  Hardware Component 插件: GazeboSimSystem / 真实驱动 / Mock    │
   └──────────────────────────────────────────────────────────────┘
```

- **Hardware Interface（硬件接口）**：每个关节暴露「命令接口」（`velocity`/`position`/`effort`）
  和「状态接口」（`position`/`velocity`）。这是与硬件对话的唯一通道；
- **Controller（控制器）**：读状态接口、写命令接口，在实时循环（通常 100~1000 Hz）里运行；
- **ControllerManager**：动态加载/激活/链式组合控制器；
- **RL 类比**：控制器 = 已经编译好的「底层策略」，硬件接口 = Gymnasium 的
  `action_space/observation_space` 约定——接口标准化，上层算法才能随意替换。

## 2. 在 URDF 中声明硬件接口：`<ros2_control>` 标签

在 diffbot 的 URDF/xacro 末尾追加（**Gazebo 仿真**用 `gz_ros2_control/GazeboSimSystem` 插件）：

```xml
<ros2_control name="DiffbotSystem" type="system">
  <hardware>
    <!-- 仿真：Gazebo 充当「硬件」；真机时换成你的驱动插件 -->
    <plugin>gz_ros2_control/GazeboSimSystem</plugin>
  </hardware>

  <joint name="left_wheel_joint">
    <command_interface name="velocity">
      <param name="min">-10</param><param name="max">10</param>  <!-- rad/s 限幅 -->
    </command_interface>
    <state_interface name="position"/>
    <state_interface name="velocity"/>
  </joint>

  <joint name="right_wheel_joint">
    <command_interface name="velocity">
      <param name="min">-10</param><param name="max">10</param>
    </command_interface>
    <state_interface name="position"/>
    <state_interface name="velocity"/>
  </joint>
</ros2_control>

<!-- 让 Gazebo 加载 ros2_control 插件，并指定控制器参数文件 -->
<gazebo>
  <plugin filename="gz_ros2_control-system" name="gz_ros2_control::GazeboSimROS2ControlPlugin">
    <parameters>$(find my_robot_bringup)/config/diffbot_controllers.yaml</parameters>
  </plugin>
</gazebo>
```

> 无 Gazebo 的纯桌面调试可用 `mock_components/GenericSystem` 插件——
> 硬件接口的「假实现」，控制器照常运行，适合 CI 与逻辑联调。

## 3. 控制器配置：diffbot_controllers.yaml

```yaml
controller_manager:
  ros__parameters:
    update_rate: 100                      # 控制循环 100 Hz
    joint_state_broadcaster:
      type: joint_state_broadcaster/JointStateBroadcaster
    diff_drive_controller:
      type: diff_drive_controller/DiffDriveController

diff_drive_controller:
  ros__parameters:
    left_wheel_names: ["left_wheel_joint"]
    right_wheel_names: ["right_wheel_joint"]
    wheel_separation: 0.35                # 轮距（必须等于 URDF 几何！）
    wheel_radius: 0.08
    odom_frame_id: odom
    base_frame_id: base_link
    enable_odom_tf: true                  # 广播 odom -> base_link（Nav2 需要）
    cmd_vel_timeout: 0.5                  # 0.5s 收不到指令就停车（安全！）
    velocity_rolling_window_size: 10
```

**三个新手高频错误**：

1. `wheel_separation` 与 URDF 不一致 → 里程计系统性漂移，导航画地图全是斜的；
2. `cmd_vel_timeout` 没设 → 上位节点崩溃后机器人永远执行最后一条指令（安全事故）；
3. 忘记 `joint_state_broadcaster` → `/joint_states` 为空，`robot_state_publisher`
   不发轮子的 TF，RViz 里轮子「掉在地上」。

## 4. 生命周期与启动：spawner

ros2_control 的控制器是**生命周期节点**：加载（load）→ 配置（configure）→ 激活（activate）。
`spawner` 一条命令完成三步：

```bash
# 启动 Gazebo（插件自动拉起 controller_manager）后：
ros2 run controller_manager spawner joint_state_broadcaster
ros2 run controller_manager spawner diff_drive_controller

# 查看状态
ros2 control list_controllers
ros2 control list_hardware_interfaces
```

**预期输出**：

```text
joint_state_broadcaster [joint_state_broadcaster/JointStateBroadcaster] active
diff_drive_controller   [diff_drive_controller/DiffDriveController] active
```

此后 `ros2 topic pub /diff_drive_controller/cmd_vel geometry_msgs/msg/TwistStamped ...`
即可驱动小车；`ros2 topic echo /diff_drive_controller/odom` 看到里程计，
`ros2 run tf2_ros tf2_echo odom base_link` 看到 TF。

## 5. 差速运动学：控制器里在算什么

`diff_drive_controller` 的核心数学（也是你 W27 要对接的接口）：

已知轮距 $L$、轮半径 $r$，机体线速度 $v$、角速度 $\omega$ 与左右轮角速度的关系：

$$
\dot\phi_{right} = \frac{v + \omega L/2}{r}, \qquad
\dot\phi_{left} = \frac{v - \omega L/2}{r}
$$

反解（里程计）：由编码器读 $\dot\phi_{left}, \dot\phi_{right}$ 积分出位姿
$(x, y, \theta)$。下面的纯 Python 模拟完整复现这个循环（本机执行）：

In [1]:
"""差速驱动：cmd_vel -> 轮速指令 -> 数值积分里程计（纯 Python，本机执行）。

模拟 diff_drive_controller 的 update() 循环 + 理想硬件执行。
"""
import numpy as np

WHEEL_RADIUS = 0.08     # r
WHEEL_SEPARATION = 0.35 # L
DT = 0.01               # 控制器 update_rate = 100 Hz

def twist_to_wheel(v, omega, r=WHEEL_RADIUS, L=WHEEL_SEPARATION):
    """控制器正解：机体速度 -> 左右轮角速度 (rad/s)。"""
    return (v - omega * L / 2) / r, (v + omega * L / 2) / r

def wheel_to_twist(wl, wr, r=WHEEL_RADIUS, L=WHEEL_SEPARATION):
    """里程计反解：左右轮角速度 -> 机体 (v, omega)。"""
    v = r * (wr + wl) / 2
    omega = r * (wr - wl) / L
    return v, omega

# --- 模拟：1s 直行 + 2s 原地左转 + 1s 直行 ---
plan = [(1.0, 1.0, 0.0), (2.0, 0.0, 0.8), (1.0, 1.0, 0.0)]  # (时长, v, omega)
x, y, th = 0.0, 0.0, 0.0
traj = [(x, y, th)]
for duration, v, w in plan:
    wl, wr = twist_to_wheel(v, w)                       # 控制器算轮速
    for _ in range(int(duration / DT)):
        v_meas, w_meas = wheel_to_twist(wl, wr)         # 从「编码器」反解
        th += w_meas * DT                                # 中点积分更准，这里用欧拉
        x += v_meas * np.cos(th) * DT
        y += v_meas * np.sin(th) * DT
        traj.append((x, y, th))

x, y, th = traj[-1]
print(f"末位姿: x={x:.3f} m, y={y:.3f} m, theta={np.rad2deg(th):.1f} deg")
print(f"手算对照: 直行 1 m → 原地转 {0.8*2:.1f} rad → 再直行 1 m，")
print(f"          理论末位姿 x={1.0 + np.cos(0.8*2):.3f}, y={np.sin(0.8*2):.3f}, theta={np.rad2deg(0.8*2):.1f} deg")
err = abs(th - 0.8 * 2)
print(f"航向积分误差: {np.rad2deg(err):.3f} deg（理想无噪声下主要来自欧拉积分）")

末位姿: x=0.971 m, y=1.000 m, theta=91.7 deg
手算对照: 直行 1 m → 原地转 1.6 rad → 再直行 1 m，
          理论末位姿 x=0.971, y=1.000, theta=91.7 deg
航向积分误差: 0.000 deg（理想无噪声下主要来自欧拉积分）


**观察**：理想条件下数值里程计与手算吻合；真实系统中编码器量化、轮胎打滑、
`wheel_separation` 标定误差都会让积分漂移——所以 Nav2 用 AMCL/SLAM 融合外部观测
修正里程计（W26），这也是为什么 `wheel_separation` 标定是现场工程师的日常。

## ✏️ 练习

### 练习 1（★，约 15 分钟）：概念连线

不用任何代码，画一张你自己机器人的 ros2_control 数据流图：从 `cmd_vel` 话题到
Gazebo 中的轮子，标出每一段经过的组件（controller / interface / hardware plugin）。
交付：图（手绘拍照或 mermaid）+ 每个组件一句话职责。

### 练习 2（★★，约 30 分钟）：mock 硬件全流程

不启动 Gazebo，仅用 `mock_components/GenericSystem` 插件 +
`ros2_control_node` 跑通 `joint_state_broadcaster` + `forward_command_controller`
（`ros-jazzy-forward-command-controller`）。交付：launch/URDF/YAML 文件 +
`ros2 control list_controllers` 全 active 的输出。

### 练习 3（★★，约 30 分钟，纯 Python）：打滑下的里程计漂移

扩展第 5 节的模拟：给左右轮真实速度各乘上 `1 + ε`，`ε ~ N(0, 0.02)`（模拟打滑/半径误差），
跑 30 s 直行指令，统计 100 次蒙特卡洛的末位姿标准差。再把 `wheel_radius` 标定值故意设错 1%，
观察系统性偏差。交付：代码 + 两组误差数据 + 结论（随机误差 vs 系统误差哪个更致命）。

### 练习 4（★★★，约 50 分钟）：Gazebo 中的完整控制器链

在 W24 的 diffbot 上加 `<ros2_control>` + `gz_ros2_control` 插件 + 本讲 YAML，
启动后 spawner 两个控制器，用 `TwistStamped` 驱动小车走 1 m × 1 m 正方形（开环），
对比 `/diff_drive_controller/odom` 轨迹与指令轨迹。交付：全部配置文件 + odom 轨迹图 +
误差归因分析（至少列出 2 个误差来源）。

## 参考答案

<details>
<summary>参考答案</summary>

**练习 1**：`cmd_vel → diff_drive_controller（运动学正解）→ velocity command interface
→ GazeboSimSystem（写入 Gazebo 关节）→ 物理仿真 → joint state（position/velocity）
→ state interface → joint_state_broadcaster → /joint_states`；
同时 diff_drive_controller 读 state interface 积分出 `/odom` 与 TF。

**练习 2**（要点）：URDF 的 `<hardware><plugin>mock_components/GenericSystem</plugin></hardware>`；
启动 `ros2 run controller_manager ros2_control_node --ros-args -p robot_description:="$(cat robot.urdf)"`，
再 spawner。`forward_command_controller` 需声明 `joints` 与 `interface_name: velocity` 参数。

**练习 3**：随机打滑 ε 导致末位姿呈扩散分布（标准差随时间增长，约 σ√t）；
半径标定错 1% 导致直线距离系统偏差 1%（30 s × 1 m/s → 偏 0.3 m），且左右轮若不等比错
还会引入航向漂移。结论：系统误差更致命——它不随平均消失，且直接破坏地图一致性；
现场靠「直线走 5 m 量实际距离」标定 `wheel_radius`，靠「原地转 10 圈」标定 `wheel_separation`。

**练习 4**：正方形开环轨迹的 odom 会呈圆角 + 终点偏移。误差来源：
(a) cmd_vel 阶跃后轮子有速度响应延迟（Gazebo 关节 PID/惯性）；
(b) 开环无反馈，角速度积分误差累积；
(c) `cmd_vel_timeout`/发布频率不齐造成的速度空窗。
这正是要用闭环（Nav2 controller 或 RL 策略）替代开环指令的原因。
</details>

## 延伸阅读

- [ros2_control 官方文档（Jazzy）](https://control.ros.org/jazzy/index.html) 与 [Getting Started](https://control.ros.org/jazzy/doc/getting_started/getting_started.html)
- [ros2_control_demos 示例仓库](https://github.com/ros-controls/ros2_control_demos)（diffbot 完整例程，强烈建议跑通 demo 2）
- [diff_drive_controller 用户文档](https://control.ros.org/jazzy/doc/ros2_controllers/diff_drive_controller/doc/userdoc.html)
- 下一讲预告：W26 在这套里程计与 TF 之上，接入 Nav2 实现自主导航。